In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Query `*.xlsx` files with SQL for free\* using Pandas and BigQuery DataFrames

In this tutorial, you'll use SQL query the [USDA wheat data](https://www.ers.usda.gov/data-products/wheat-data), which is distributed as files in the [Excel (.xlsx) Extensions to the Office Open XML SpreadsheetML file format](https://learn.microsoft.com/en-us/openspecs/office_standards/ms-xlsx/2c5dee00-eff2-4b22-92b6-0738acd4475e). Because of open source packages like Jupyter, Pandas, and BigQuery DataFrames (aka BigFrames) and the [BigQuery sandbox](https://docs.cloud.google.com/bigquery/docs/sandbox), you should be able to follow all of the steps in this guide for free\* and without a credit card. 

_\*See the [BigQuery sandbox](https://docs.cloud.google.com/bigquery/docs/sandbox) documentation for limitations._

BigQuery DataFrames aka BigFrames is an open source Python library offered by Google. BigFrames scales Python data processing by transpiling common Python data science APIs to BigQuery SQL. You can read more about BigFrames in the [official introduction to BigFrames](https://dataframes.bigquery.dev/user_guide/index.html) and can refer to the [public git repository for BigFrames](https://github.com/googleapis/google-cloud-python/tree/main/packages/bigframes).

Last year, Google introduced [SQL cells in Colab Enterprise notebooks](https://docs.cloud.google.com/colab/docs/sql-cells), which was a collaboration across several teams, including the BigQuery DataFrames team. Now, with the [%%bqsql cell magics](https://dataframes.bigquery.dev/notebooks/getting_started/magics.html) available in BigQuery DataFrames (aka BigFrames), this same functionality is available to all Jupyter notebook users, whether you're in Colab, Jupyter Lab, or a notebook in VS Code. These magics use BigQuery to query a table or even a local pandas DataFrame.


# Getting Started

To get started,

1. Enable the [BigQuery sandbox](https://docs.cloud.google.com/bigquery/docs/sandbox). Make note of your Google Cloud project ID.

2. Set up a local Python development environment (see: [Setting up a Python development environment](https://docs.cloud.google.com/python/docs/setup)) for Google Cloud.

3. Create and activate a venv to isolate Python dependencies. \
 \
On Linux or macOS, use these commands (update to your preferred Python version): \



```
python3.12 -m venv ~/venv
. ~/venv/bin/activate
```


4. Install the Jupyter, bigframes, and python-calamine packages \



```
pip install --upgrade jupyterlab bigframes python-calamine
```


5. Start Jupyter Lab.


```
jupyter lab
```


6. Open a web browser to the URL listed in the output. It will be something like http://localhost:8888/lab?token=somesupersecretvaluehere .

7. Create a new notebook using the Jupyter Lab UI.


In [2]:
%pip install python-calamine pandas bigframes

  Using cached pandas-2.3.3-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
Using cached pandas-2.3.3-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.3 MB)
  Attempting uninstall: pandas
    Found existing installation: pandas 3.0.2
    Uninstalling pandas-3.0.2:
      Successfully uninstalled pandas-3.0.2
Note: you may need to restart the kernel to use updated packages.


## Accessing the data

In this tutorial, you'll analyze the [USDA wheat data](https://www.ers.usda.gov/data-products/wheat-data). Use the requests package to download the data to a temporary file.

In [1]:

import tempfile

import requests

url = "https://www.ers.usda.gov/media/5706/wheat-data-all-years.xlsx?v=52690"

tmp = tempfile.NamedTemporaryFile(delete=True)

with requests.get(url, stream=True) as r:
    r.raise_for_status()
    for chunk in r.iter_content(chunk_size=8192):
        tmp.write(chunk)

tmp.flush()
tmp.seek(0)

0


When working with SQL, use the pyarrow dtype_backend for more consistent handling of NULL values. The Table05 sheet provides annual data:

In [2]:

import pandas as pd

df = pd.read_excel(
    tmp,
    sheet_name="Table05",
    dtype_backend="pyarrow",
    engine="calamine",
    header=1,  # Skip the first row.
)
tmp.close()
df

,Marketing year 1/,Time period,Beginning stocks,Production,Imports 2/,Total supply 3/,Food use,Seed use,Feed and residual use,Total domestic use 3/,Exports 2/,Total disappearance 3/,Ending stocks
0,1950/51,MY Jun-May,496.0,1019.0,11.0,1526.0,580.0,--,109.0,689.0,345.0,1034.0,492.0
1,1951/52,MY Jun-May,492.0,988.0,30.0,1510.0,585.0,--,110.0,695.0,485.0,1180.0,330.0
2,1952/53,MY Jun-May,330.0,1306.0,24.0,1660.0,578.0,--,78.0,656.0,332.0,988.0,672.0
3,1953/54,MY Jun-May,672.0,1173.0,6.0,1851.0,556.0,--,87.0,643.0,214.0,857.0,994.0
4,1954/55,MY Jun-May,994.0,984.0,3.0,1981.0,552.0,--,53.0,605.0,267.0,872.0,1109.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
281,1/ June–May. Latest data may be preliminary or...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
282,2/ Includes flour and selected other products ...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
283,3/ Totals may not add due to rounding.,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
284,"Source: USDA, Economic Research Service, based...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


Rename the columns to be more SQL-friendly. BigQuery supports [flexible column names](https://docs.cloud.google.com/bigquery/docs/schemas#flexible-column-names), which allows most unicode characters, but some special characters such as "\" and "/" aren't allowed.

In [3]:
df.columns = [name.replace("/", "") for name in df.columns]
df

,Marketing year 1,Time period,Beginning stocks,Production,Imports 2,Total supply 3,Food use,Seed use,Feed and residual use,Total domestic use 3,Exports 2,Total disappearance 3,Ending stocks
0,1950/51,MY Jun-May,496.0,1019.0,11.0,1526.0,580.0,--,109.0,689.0,345.0,1034.0,492.0
1,1951/52,MY Jun-May,492.0,988.0,30.0,1510.0,585.0,--,110.0,695.0,485.0,1180.0,330.0
2,1952/53,MY Jun-May,330.0,1306.0,24.0,1660.0,578.0,--,78.0,656.0,332.0,988.0,672.0
3,1953/54,MY Jun-May,672.0,1173.0,6.0,1851.0,556.0,--,87.0,643.0,214.0,857.0,994.0
4,1954/55,MY Jun-May,994.0,984.0,3.0,1981.0,552.0,--,53.0,605.0,267.0,872.0,1109.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
281,1/ June–May. Latest data may be preliminary or...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
282,2/ Includes flour and selected other products ...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
283,3/ Totals may not add due to rounding.,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
284,"Source: USDA, Economic Research Service, based...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


You can use pandas syntax to filter rows.


In [ ]:
full_rows = df[~df['Beginning stocks'].isna()]
full_rows


# Using BigQuery SQL magics (%%bqsql)

The BigQuery DataFrames (aka BigFrames) library provides a %%bqsql magic, which can query local pandas or BigFrames DataFrames, as well as anything supported by the BigQuery query engine, such as parquet / iceberg, CSV files in GCS, and BigQuery tables. To enable the magic, use the %load_ext magic.


In [ ]:
%load_ext bigframes


To ensure the correct Google Cloud project is billed for query usage, including free tier usage, configure the project ID used by the magics. If not set, the default project is discovered from your environment, such as the one associated with your application default credentials.


In [ ]:
import bigframes.pandas as bpd

bpd.options.bigquery.project = "your-project-id"


Now that the extension is loaded, you can use the %%bqsql magics to query the DataFrame created in the previous steps.


In [ ]:
%%bqsql
SELECT * FROM {full_rows}


You should see the results from full_rows.


# Transforming the data with SQL

The %%bqsql magics take a destination variable argument, which saves the results as a BigFrames DataFrame. This can be used to incrementally apply operations in both SQL and Python.

First, limit the data to yearly data and save the results to the "yearly" variable.


In [ ]:
%%bqsql yearly
SELECT *
FROM {full_rows}
WHERE STARTS_WITH(`Time period`, 'MY')


The "Marketing year 1" column isn't as useful as it could be because it is still a string. Transform it to a time series using SQL.


In [ ]:
%%bqsql timeseries
SELECT
  * EXCEPT (`Marketing year 1`),
  TIMESTAMP(CONCAT(
    REGEXP_EXTRACT(`Marketing year 1`, r'([0-9]+)\/'),
    '-01-01')) AS `year`
FROM {yearly}


# Visualizing the data

BigFrames supports most pandas operations, including several visualization methods. By setting a timestamp column to the index of the DataFrame, visualization becomes easier to understand.


In [ ]:
timeseries.set_index('year').sort_index().plot.line()


Alternatively, convert the DataFrame to pandas for further integration with other libraries.


In [ ]:
pddf = timeseries.set_index('year').sort_index().to_pandas()
pddf


# Conclusion

By leveraging BigFrames, you can combine the best of both worlds: the expressive power of SQL and the versatile ecosystem of Python. This approach not only improves readability but also provides the opportunity to scale your data processing to handle massive datasets directly in BigQuery by swapping out a pandas DataFrame in these examples with a BigQuery DataFrame.


# Next Steps

Another way to use BigQuery features on pandas DataFrames is through the BigQuery pandas extension. For example, call any of the community BigQuery functions in [BigQuery Utils](https://github.com/GoogleCloudPlatform/bigquery-utils/tree/master/udfs#bigquery-udfs), [BigFunctions](https://unytics.io/bigfunctions/bigfunctions/#function-categories), [CARTO Analytics Toolbox for BigQuery](https://docs.carto.com/data-and-analysis/analytics-toolbox-for-bigquery), and more by using the DataFrame.bigquery.sql_scalar(...) accessor.


In [ ]:
import pandas as pd

import bigframes.pandas as bpd  # registers the bigquery accessor

data = {
    'text1': [
        'apple',
        'banana',
        'orange',
        'grape',
        'strawberry',
        'blueberry',
        'raspberry',
        'pineapple'
    ],
    'text2': [
        'aple',
        'bandana',
        'orenge',
        'grpe',
        'straaawberry',
        'bluebery',
        'rasery',
        'pinapple'
    ]
}

df = pd.DataFrame(data)

bpd.options.bigquery.project = "your-project-id"

df.bigquery.sql_scalar("bqutil.fn.cw_editdistance({text1}, {text2})")


BigQuery sandbox offers powerful, scalable analytics, but some features aren't supported, such as BigQuery Machine Learning. Connect a billing account to your project to use powerful features such as the AI.FORECAST function, which can predict time series data using Google's foundational models.

The BigFrames team would love to hear from you. If you would like to reach out, please send an email to: [bigframes-feedback@google.com](mailto:bigframes-feedback@google.com) or by filing an issue at the[ open source BigFrames repository](https://github.com/googleapis/google-cloud-python/issues). To receive updates about BigFrames, subscribe to the [BigFrames email list](https://docs.google.com/forms/d/10EnDyYdYUW9HvelHYuBRC8L3GdGVl3rX0aroinbRZyc/edit?resourcekey=0-QUsnpzF91gm9hsp04rSA6Q).